In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [22]:
from src.ml.model import AKIModel
from src.ml.dataset import AKIDataset, AKINormalizer, aki_collate_fn
from src.ml.trainer import train, evaluate
from src.ml.inference import infer

import json

In [13]:
train_ds = AKIDataset("../data/training.csv")

In [35]:
train_ds.save_normalizers("../config/normalizers.config")

In [29]:
normalizer = AKINormalizer.load("../config/normalizers.config")
test_ds = AKIDataset(file_name="../data/test.csv", normalizer=normalizer)

c:\Users\andre\OneDrive\Desktop\Projects\swemls\src\ml\dataset.py:123: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ages = df["age"].tolist()


In [30]:
model = AKIModel()

In [21]:
history = train(
    model,
    train_ds,
    test_ds,
    aki_collate_fn,
    num_epochs=15,
    batch_size=32,
    lr=1e-3,
    checkpoint_dir="../checkpoints",
    early_stopping_patience=3,
    save_best=True,
)

c:\Users\andre\OneDrive\Desktop\Projects\swemls\.env\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
Epoch 1/15:  70%|██████▉   | 160/229 [00:04<00:01, 38.20batch/s]


KeyboardInterrupt: 

In [63]:
json.dump(history, open("../logs/history.json", "w"))

In [31]:
import torch

model.load_state_dict(torch.load("../checkpoints/best_model.pth"))
metrics = evaluate(model, test_ds, aki_collate_fn)

C:\Users\andre\AppData\Local\Temp\ipykernel_27584\647449644.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../checkpoints/best_model.p

In [32]:
metrics

{'fp': 11,
 'tp': 1557,
 'fn': 7,
 'tn': 5827,
 'precision': 0.9929846938775511,
 'recall': 0.9955242966751918,
 'f3': 0.9952697509795453}

In [34]:
import pandas as pd

age = 45
creatinine = [89.86]
dates = [pd.to_datetime("2024-01-01 17:54:00")]

infer(model, normalizer, age, creatinine, dates)

0